# 05 — Rollout: avaliação do modelo treinado no pipeline HDF5

Roda no simulador de verdade (LiberoEnv) o checkpoint treinado por
`04_treino_hdf5.ipynb` e mede taxa de sucesso -- o teste que realmente
importa, não só a curva de loss. Ver `docs/hdf5_migration.md` para o
contexto completo da migração.

**Diferenças em relação a `02_rollout_libero.ipynb`** (que é single-task,
sem linguagem):
- Roda **uma tarefa por vez**, em loop, com fusão de linguagem (FiLM/
  token/cross_attn conforme `CONFIG` abaixo).
- **`LiberoEnv` renderiza a 128×128`**, não 256×256 (o default da lib) --
  o modelo foi treinado em HDF5 nativo 128×128; avaliar a 256×256 daria
  ao modelo uma grade de tokens de imagem que ele nunca viu no treino.
- Os **normalizadores** vêm do `meta.json` computado sobre o MESMO
  subconjunto HDF5 usado no treino (`compute_stats`, em
  `data/hdf5_libero.py`), não das stats globais do `lerobot/libero` --
  elas diferem ligeiramente (o HDF5 usa mais episódios por tarefa que o
  lerobot). Por isso este notebook baixa de novo o mesmo `PILOT_LIMIT` de
  tarefas antes de avaliar: `compute_stats` é determinístico dado o mesmo
  conjunto de arquivos, então os stats recalculados aqui são IDÊNTICOS
  aos usados no treino.
- A **suite oficial** de cada tarefa (`libero_object`/`libero_goal`/etc,
  necessária pra montar o `task_suite`/`task_id` do simulador) vem de
  `configs/task_mapping_libero40.json` -- não é a mesma coisa que
  `cfg["task_suite_name"]` (`"libero_40_mixed"`, um pseudo-nome só pro
  filtro de dados, não uma suite real do benchmark).

**Ajuste `PILOT_LIMIT` abaixo para bater com o que você treinou** (mesmo
valor usado em `04_treino_hdf5.ipynb`) -- ele decide tanto quais tarefas
são avaliadas quanto os stats de normalização recalculados.

Precisa de `lerobot[libero]` (o simulador de verdade -- diferente do
treino, aqui não tem como evitar essa dependência pesada).

## 1. Repositório e ambiente

In [ ]:
!git clone -b correcoes-set2026 https://github.com/rafaelheydt/act-lang.git
%cd act-lang

!pip install -q -e ".[hdf5,language]" "lerobot[libero]"

## 2. Config: ajustar `device_index` para o Colab

Mesmo ajuste do notebook de treino -- `device_index: 1` é da máquina do
CEPEDI; `None` deixa o `pick_device` escolher sozinho.

In [ ]:
!sed -i 's/"device_index": 1,/"device_index": None,  # None = auto -- era 1 p\/ CEPEDI (2 GPUs)/' configs/libero_40tasks_language.py
!grep -n "device_index" configs/libero_40tasks_language.py

## 3. Normalizadores (mesmo subconjunto piloto do treino)

`PILOT_LIMIT` precisa bater com o valor usado em `04_treino_hdf5.ipynb` --
decide quais tarefas foram treinadas E reproduz os stats exatos de
normalização (determinísticos dado o mesmo conjunto de arquivos).

In [ ]:
PILOT_LIMIT = 3  # mesmo valor usado no treino -- None se treinou as 40 completas
DATA_DIR = "/content/libero_hdf5"

import subprocess, sys
cmd = [sys.executable, "-u", "scripts/download_libero_hdf5.py", "--out", DATA_DIR]
if PILOT_LIMIT is not None:
    cmd += ["--limit", str(PILOT_LIMIT)]
print("rodando:", " ".join(cmd))
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end="")
proc.wait()
assert proc.returncode == 0, "download falhou -- veja o log acima"

import torch
from act_lang.data.hdf5_libero import load_meta, build_normalizers_from_meta
from act_lang.utils.runtime import describe_devices, pick_device, is_colab, get_checkpoint_dir

print(describe_devices())

from configs.libero_40tasks_language import CONFIG_FILM as cfg  # troque se usou outro mecanismo
device = pick_device(preferred_index=cfg.get("device_index"))
print(f"usando: {device}")

meta = load_meta(DATA_DIR)
state_norm, action_norm = build_normalizers_from_meta(meta, device)

# fps de referência dos vídeos do LIBERO (metadado leve, não baixa o dataset de vídeo inteiro)
from lerobot.datasets.lerobot_dataset import LeRobotDatasetMetadata
lerobot_meta = LeRobotDatasetMetadata("lerobot/libero")
video_fps = int(lerobot_meta.fps)
print(f"normalizadores prontos | video_fps={video_fps}")

## 4. Carregar o checkpoint treinado

Mesmo `CHECKPOINT_DIR` (sufixo `_hdf5128`) que `04_treino_hdf5.ipynb`
usou pra salvar -- monta o Drive só pra ler o checkpoint (o dataset fica
no disco local, seção 3 acima).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
CHECKPOINT_DIR = Path(f"/content/drive/MyDrive/{cfg['experiment_name']}_hdf5128")
print("checkpoints em:", CHECKPOINT_DIR)

from act_lang.models.act import ACT
from act_lang.models.backbone import freeze_batchnorm
from act_lang.models.fusion import build_fusion
from act_lang.training.checkpoints import load_checkpoint

model = ACT(
    action_dim=cfg["action_dim"], state_dim=cfg["state_dim"], d_model=cfg["d_model"],
    latent_dim=cfg["latent_dim"], chunk_size=cfg["chunk_size"], n_cameras=cfg["n_cameras"],
    n_encoder_layers=cfg["n_encoder_layers"], n_decoder_layers=cfg["n_decoder_layers"],
    n_heads=cfg["n_heads"], dropout=cfg["dropout"], pretrained_backbone=False,
    decoder_style=cfg["decoder_style"], fusion=build_fusion(cfg["fusion_type"], cfg["d_model"]),
)
if cfg["freeze_bn"]:
    freeze_batchnorm(model.vision_backbone)
model = model.to(device)

best = sorted(CHECKPOINT_DIR.glob("best_epoch*.pt"))[-1]  # ou aponte o arquivo à mão
print(f"carregando: {best.name}")
next_epoch, _ = load_checkpoint(best, model, device=device)
model.eval()
print(f"checkpoint da época {next_epoch - 1}")

## 5. Rollout por tarefa

Uma tarefa por vez: resolve a suite oficial + `task_id` via
`configs/task_mapping_libero40.json`, monta o `LiberoEnv` a 128×128 e
roda `rollout_libero` (temporal ensembling, mesma implementação de
`02_rollout_libero.ipynb`).

In [ ]:
import json
import numpy as np
from libero.libero import benchmark
from lerobot.envs.libero import LiberoEnv
from act_lang.eval.rollout_libero import rollout_libero

task_mapping = json.loads(Path("configs/task_mapping_libero40.json").read_text(encoding="utf-8"))["tasks"]
all_tasks_sorted = sorted(task_mapping.items())
trained_tasks = all_tasks_sorted[:PILOT_LIMIT] if PILOT_LIMIT is not None else all_tasks_sorted
print(f"{len(trained_tasks)} tarefa(s) treinada(s):")
for t, v in trained_tasks:
    print(" -", t, f"(suite oficial: {v['suite']})")

video_root = "/content/libero_rollouts" if is_colab() else str(CHECKPOINT_DIR / "rollouts")

all_results = {}
for task_text, info in trained_tasks:
    suite_name = info["suite"]
    task_suite = benchmark.get_benchmark_dict()[suite_name]()
    task_id = next(
        i for i in range(len(task_suite.tasks))
        if task_suite.get_task(i).language == task_text
    )
    env = LiberoEnv(
        task_suite=task_suite, task_id=task_id, task_suite_name=suite_name,
        control_mode="relative", render_mode="rgb_array",
        obs_type="pixels_agent_pos",
        observation_width=128, observation_height=128,  # bate com o treino (HDF5 nativo)
    )
    task_video_dir = f"{video_root}/{task_text[:40].replace(' ', '_')}"
    print(f"\n=== {task_text} (suite={suite_name}, task_id={task_id}) ===")
    results = rollout_libero(
        model, env, state_norm, action_norm, device,
        n_episodes=cfg["rollout_n_episodes"], m=cfg["rollout_m"],
        max_steps=cfg["rollout_max_steps"], video_dir=task_video_dir,
        video_fps=video_fps, task_text=task_text,
    )
    env.close()
    all_results[task_text] = results
    sr = np.mean([r["success"] for r in results])
    print(f"taxa de sucesso: {sr:.1%}")

## 6. Resumo

In [ ]:
import pandas as pd

rows = [
    {
        "tarefa": task_text,
        "n_episodios": len(results),
        "taxa_sucesso": np.mean([r["success"] for r in results]),
    }
    for task_text, results in all_results.items()
]
df = pd.DataFrame(rows)
print(df.to_string(index=False))

overall = np.mean([r["success"] for results in all_results.values() for r in results])
print(f"\nTaxa de sucesso geral ({len(all_results)} tarefa(s)): {overall:.1%}")

## 7. Ver um episódio

In [ ]:
from IPython.display import Video, display

primeira_tarefa, primeiros_resultados = next(iter(all_results.items()))
r0 = primeiros_resultados[0]
tag = "sucesso" if r0["success"] else "falha"
task_video_dir = f"{video_root}/{primeira_tarefa[:40].replace(' ', '_')}"
display(Video(f"{task_video_dir}/ep00_state{r0['init_state_id']:02d}_{tag}.mp4", embed=True, width=700))